In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned_loan_data.csv")
print(df.shape)
df.head()

(149999, 16)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,deliquency_data_error,income_was_missing,TotalTimesLate,IncomePerDependent,TotalCreditLines
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0,0,0,2,3040.0,19
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,0,0,0,1300.0,4
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,0,0,2,3042.0,2
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,0,0,0,3300.0,5
4,0,0.907239,49,1,0.024926,23000.0,7,0,1,0,0.0,0,0,1,23000.0,8


In [2]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

In [6]:
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

mlflow.set_experiment("loan-default-ensembles")

models = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    with mlflow.start_run(run_name=f"{name}_baseline"):
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, probs)
        results[name] = auc

        mlflow.log_param("model_type", name)
        mlflow.log_metric("val_auc", auc)

        print(f"{name}: AUC = {auc:.4f}")

RandomForest: AUC = 0.8435
AdaBoost: AUC = 0.8497
GradientBoosting: AUC = 0.8689


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

param_grid = {
    'n_estimators': [50, 100, 200, 400],
    'max_depth': [3, 5, 10, None],
    'max_features': ['sqrt', 0.3, 0.6, 1.0],
    'min_samples_leaf': [1, 5, 20, 50]
}

sensitivity_results = {}
for param_name, values in param_grid.items():
    aucs = []
    for v in values:
        params = {'random_state': 42, param_name: v}
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, probs)
        aucs.append(auc)
        print(f"{param_name}={v}: AUC={auc:.4f}")
    swing = max(aucs) - min(aucs)
    sensitivity_results[param_name] = swing
    print(f"-->{param_name} swing: {swing:.4f}\n")
    
print("Ranked by impact(biggest swing first):")
for k, v in sorted(sensitivity_results.items(), key=lambda x: -x[1]):
    print(f"{k}: {v:.4f}")


n_estimators=50: AUC=0.8355
n_estimators=100: AUC=0.8435
n_estimators=200: AUC=0.8474
n_estimators=400: AUC=0.8504
-->n_estimators swing: 0.0148

max_depth=3: AUC=0.8555
max_depth=5: AUC=0.8607
max_depth=10: AUC=0.8679
max_depth=None: AUC=0.8435
-->max_depth swing: 0.0244

max_features=sqrt: AUC=0.8435
max_features=0.3: AUC=0.8432
max_features=0.6: AUC=0.8411
max_features=1.0: AUC=0.8404
-->max_features swing: 0.0031

min_samples_leaf=1: AUC=0.8435
min_samples_leaf=5: AUC=0.8603
min_samples_leaf=20: AUC=0.8654
min_samples_leaf=50: AUC=0.8669
-->min_samples_leaf swing: 0.0234

Ranked by impact(biggest swing first):
max_depth: 0.0244
min_samples_leaf: 0.0234
n_estimators: 0.0148
max_features: 0.0031


In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

results_list = []

# n_estimators sweep
aucs = []
for v in [50, 100, 200, 400]:
    model = AdaBoostClassifier(n_estimators=v, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"n_estimators={v}: AUC={auc:.4f}")
print(f"--> n_estimators swing: {max(aucs) - min(aucs):.4f}\n")

# learning_rate sweep
aucs = []
for v in [0.01, 0.1, 0.5, 1.0, 2.0]:
    model = AdaBoostClassifier(learning_rate=v, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"learning_rate={v}: AUC={auc:.4f}")
print(f"--> learning_rate swing: {max(aucs) - min(aucs):.4f}\n")

# base estimator depth sweep
aucs = []
for v in [1, 2, 3, 5]:
    model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=v), random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"base_depth={v}: AUC={auc:.4f}")
print(f"--> base_depth swing: {max(aucs) - min(aucs):.4f}\n")

n_estimators=50: AUC=0.8497
n_estimators=100: AUC=0.8598
n_estimators=200: AUC=0.8642
n_estimators=400: AUC=0.8653
--> n_estimators swing: 0.0156

learning_rate=0.01: AUC=0.6957
learning_rate=0.1: AUC=0.8545
learning_rate=0.5: AUC=0.8553
learning_rate=1.0: AUC=0.8497
learning_rate=2.0: AUC=0.5000
--> learning_rate swing: 0.3553

base_depth=1: AUC=0.8497
base_depth=2: AUC=0.8642
base_depth=3: AUC=0.8624
base_depth=5: AUC=0.8618
--> base_depth swing: 0.0146



In [9]:
from sklearn.ensemble import GradientBoostingClassifier

# n_estimators sweep
aucs = []
for v in [50, 100, 200, 400]:
    model = GradientBoostingClassifier(n_estimators=v, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"n_estimators={v}: AUC={auc:.4f}")
print(f"--> n_estimators swing: {max(aucs) - min(aucs):.4f}\n")

# learning_rate sweep
aucs = []
for v in [0.01, 0.1, 0.5, 1.0, 2.0]:
    model = GradientBoostingClassifier(learning_rate=v, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"learning_rate={v}: AUC={auc:.4f}")
print(f"--> learning_rate swing: {max(aucs) - min(aucs):.4f}\n")

# max_depth sweep
aucs = []
for v in [1, 2, 3, 5, 10]:
    model = GradientBoostingClassifier(max_depth=v, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    aucs.append(auc)
    print(f"max_depth={v}: AUC={auc:.4f}")
print(f"--> max_depth swing: {max(aucs) - min(aucs):.4f}\n")

n_estimators=50: AUC=0.8669
n_estimators=100: AUC=0.8689
n_estimators=200: AUC=0.8699
n_estimators=400: AUC=0.8695
--> n_estimators swing: 0.0030

learning_rate=0.01: AUC=0.8504
learning_rate=0.1: AUC=0.8689
learning_rate=0.5: AUC=0.8669
learning_rate=1.0: AUC=0.7294
learning_rate=2.0: AUC=0.5937
--> learning_rate swing: 0.2751

max_depth=1: AUC=0.8634
max_depth=2: AUC=0.8674
max_depth=3: AUC=0.8689
max_depth=5: AUC=0.8695
max_depth=10: AUC=0.8588
--> max_depth swing: 0.0107

